<a href="https://colab.research.google.com/github/Denis2054/RAG-Driven-Generative-AI/blob/main/Chapter02/2_Embeddings_vector_store_Karan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Embedding-Based Retrieval with Activeloop and OpenAI

Copyright 2024 Denis Rothman

This second component of the RAG pipeline transforms the prepared data by the first component of the pipeline into embeddings and stores the vectors obtained in the vector store.

# Installing the environment

*First run the following cells and restart Google Colab session if prompted. Then run the notebook again cell by cell to explore the code.*

In [2]:
!uv pip install deeplake==3.9.18
import deeplake

Using Python 3.11.15 environment at: /home/mahmoud/frontend-projects/practise-projects/rag-driven-generative-ai/.venv
Checked 1 package in 24ms


Mount a drive or implement the method that best fits your project to retrieve API tokens.

In [9]:
!uv pip install python-dotenv
import os
from dotenv import load_dotenv
load_dotenv()

Using Python 3.11.15 environment at: /home/mahmoud/frontend-projects/practise-projects/rag-driven-generative-ai/.venv
Checked 1 package in 1ms


True

grequests.py contains a function to download files from GitHub

In [ ]:
#GitHub grequests.py
#Script to download files from the GitHub repository.
#The private token will be removed from this function when the repository goes public.

import subprocess

# Define your private token as a variable
url = "https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI/main/commons/grequests.py"
output_file = "grequests.py"

# Prepare the curl command using the private token
curl_command = [
    "curl",
    "-H", f"Authorization: token {private_token}",
    "-o", output_file,
    url
]

# Execute the curl command
try:
    subprocess.run(curl_command, check=True)
    print("Download successful.")
except subprocess.CalledProcessError:
    print("Failed to download the file.")


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

Download successful.


100    14  100    14    0     0     40      0 --:--:-- --:--:-- --:--:--    40


In [11]:
!uv pip install openai==1.40.3

Using Python 3.11.15 environment at: /home/mahmoud/frontend-projects/practise-projects/rag-driven-generative-ai/.venv
Checked 1 package in 3ms


In [12]:
#Retrieving and setting the OpenAI API key
import os
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

In [13]:
#Retrieving and setting the Activeloop API token
import os
ACTIVELOOP_TOKEN = os.getenv("ACTIVELOOP_TOKEN")
os.environ['ACTIVELOOP_TOKEN'] = ACTIVELOOP_TOKEN

# Embedding and Storage: populating the vector store

## Downloading and preparing the data

In [14]:
from grequests import download
source_text = "llm.txt"

directory = "Chapter02"
filename = "llm.txt"
download(directory, filename, private_token)


TypeError: download() takes 2 positional arguments but 3 were given

In [15]:
# Open the file and read the first 20 lines
with open('llm.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()
    # Print the first 20 lines
    for line in lines[:20]:
        print(line.strip())

Investigation of outer space For broader coverage of this topic, see Exploration . Buzz Aldrin taking a core sample of the Moon during the Apollo 11 mission Self-portrait of Curiosity rover on Mars 's surface Part of a series on Spaceflight History History of spaceflight Space Race Timeline of spaceflight Space probes Lunar missions Mars missions Applications Communications Earth observation Exploration Espionage Military Navigation Colonization Habitation Exploration Telescopes Tourism Spacecraft Robotic spacecraft Satellite Space probe Cargo spacecraft Crewed spacecraft Apollo Lunar Module Space capsules Space Shuttle Space stations Spaceplanes Vostok Space launch Spaceport Launch pad Expendable and reusable launch vehicles Escape velocity Non-rocket spacelaunch Spaceflight types Sub-orbital Orbital Interplanetary Interstellar Intergalactic List of space organizations Space agencies Space forces Companies Spaceflight portal v t e Space exploration is the physical investigation of out

Chunking the data

In [16]:
with open("llm.txt", 'r') as f:
    text = f.read()

CHUNK_SIZE = 1000
chunked_text = [text[i:i+CHUNK_SIZE] for i in range(0,len(text), CHUNK_SIZE)]

## Verifying if the vector store exists or create it

If vector store doesn't exist, the following code will create it and display a message.

If the vectore store exists, only the "Vector store exists" message will be displayed.

Deep Lake

**Replace `hub://mahmoudkamalfci/my_table` by your organization and dataset name**

In [19]:
vector_store_path = "al://mahmoudkamal01099s-organization/first-workspace/my_table"

In [20]:
from deeplake.core.vectorstore.deeplake_vectorstore import VectorStore
import deeplake.util

try:
    # Attempt to load the vector store
    vector_store = VectorStore(path=vector_store_path)
    print("Vector store exists")
except FileNotFoundError:
    print("Vector store does not exist. You can create it.")
    # Code to create the vector store goes here
    create_vector_store=True


Vector store exists


## The embedding function

In [ ]:
def embedding_function(texts, model="text-embedding-3-small"):
   if isinstance(texts, str):
       texts = [texts]
   texts = [t.replace("\n", " ") for t in texts]
   return [data.embedding for data in openai.embeddings.create(input = texts, model=model).data]

## Adding data to the vector store

In [ ]:
add_to_vector_store=True

In [ ]:
if add_to_vector_store == True:
    with open(source_text, 'r') as f:
        text = f.read()
        CHUNK_SIZE = 1000
        chunked_text = [text[i:i+1000] for i in range(0, len(text), CHUNK_SIZE)]

vector_store.add(text = chunked_text,
              embedding_function = embedding_function,
              embedding_data = chunked_text,
              metadata = [{"source": source_text}]*len(chunked_text))


# Vector Store information

Summary

In [ ]:
# Print the summary of the Vector Store
print(vector_store.summary())

Visualize

Online:
https://app.activeloop.ai/datasets/mydatasets/

In [ ]:
ds = deeplake.load(vector_store_path)

Dataset size

In [ ]:
#Estimates the size in bytes of the dataset.
ds_size=ds.size_approx()

In [ ]:
# Convert bytes to megabytes and limit to 5 decimal places
ds_size_mb = ds_size / 1048576
print(f"Dataset size in megabytes: {ds_size_mb:.5f} MB")

# Convert bytes to gigabytes and limit to 5 decimal places
ds_size_gb = ds_size / 1073741824
print(f"Dataset size in gigabytes: {ds_size_gb:.5f} GB")